### STEP 1: Dimension Tables

### 1. dim_customer

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_customer = spark.table("novocart.silver.customers_cleaned") \
    .withColumn("customer_key", monotonically_increasing_id())

dim_customer = dim_customer.select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email",
    "country_code",
    "channel",
    "registration_date"
)

dim_customer.write.mode("overwrite").saveAsTable("novocart.gold.dim_customer")
## OPTIMIZATION
spark.sql("""
OPTIMIZE novocart.gold.dim_customer
ZORDER BY (customer_key)
""")

### 2. dim_product

In [0]:
dim_product = spark.table("novocart.silver.products_cleaned") \
    .withColumn("product_key", monotonically_increasing_id())

dim_product = dim_product.select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "price",
    "currency",
    "country_code"
)

dim_product.write.mode("overwrite").saveAsTable("novocart.gold.dim_product")

### 3. dim_date

In [0]:
from pyspark.sql.functions import year, month, dayofmonth

orders = spark.table("novocart.silver.orders_cleaned")

dim_date = orders.select("order_date").distinct() \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("day", dayofmonth("order_date"))

dim_date = dim_date.withColumn("date_key", monotonically_increasing_id())

dim_date.write.mode("overwrite").saveAsTable("novocart.gold.dim_date")

### 4. dim_channel

In [0]:
dim_channel = spark.table("novocart.silver.orders_cleaned") \
    .select("channel").distinct() \
    .withColumn("channel_key", monotonically_increasing_id())

dim_channel.write.mode("overwrite").saveAsTable("novocart.gold.dim_channel")

### 5. dim_country

In [0]:
dim_country = spark.table("novocart.silver.orders_cleaned") \
    .select("country_code").distinct() \
    .withColumn("country_key", monotonically_increasing_id())

dim_country.write.mode("overwrite").saveAsTable("novocart.gold.dim_country")

### STEP 2: FACT TABLE

In [0]:
from pyspark.sql.functions import col

orders = spark.table("novocart.silver.orders_cleaned").alias("orders")
order_items = spark.table("novocart.silver.order_items_cleaned").alias("order_items")
dim_customer = spark.table("novocart.gold.dim_customer").alias("dim_customer")
dim_product = spark.table("novocart.gold.dim_product").alias("dim_product")
dim_date = spark.table("novocart.gold.dim_date").alias("dim_date")
dim_channel = spark.table("novocart.gold.dim_channel").alias("dim_channel")
dim_country = spark.table("novocart.gold.dim_country").alias("dim_country")
currency = spark.table("novocart.silver.exchange_rates_cleaned").alias("currency")

# Join orders + order_items
fact = orders.join(order_items, "order_id")

# Join dimensions
fact = fact.join(dim_customer, "customer_id", "left") \
           .join(dim_product, "product_id", "left") \
           .join(dim_date, col("orders.order_date") == col("dim_date.order_date"), "left") \
           .join(dim_channel, "channel", "left") \
           .join(dim_country, "country_code", "left")

# Currency conversion
fact = fact.join(currency, col("orders.currency") == col("currency.currency_code"), "left") \
           .withColumn("line_total_usd", col("line_total") * col("exchange_rate_to_usd"))

# Final fact table
fact_sales = fact.select(
    "order_id",
    "order_item_id",
    "customer_key",
    "product_key",
    "date_key",
    "channel_key",
    "country_key",
    "quantity",
    "unit_price",
    "line_total",
    "line_total_usd"
)

fact_sales.write.mode("overwrite").saveAsTable("novocart.gold.fact_sales")

## OPTIMIZATION

In [0]:
# Optimize Fact Table
spark.sql("""
OPTIMIZE novocart.gold.fact_sales
ZORDER BY (customer_key, product_key, order_id)
""")